In [36]:
import subprocess
packages = [
    "pandas", "numpy", "scikit-learn",
    "matplotlib", "seaborn", "nltk",
    "spacy", "beautifulsoup4", "joblib",
    "flask", "flask-cors", "imbalanced-learn",
    "PyMuPDF", "python-docx"
]
for pkg in packages:
    subprocess.run(["pip", "install", pkg])
print("✅ All installed!")

✅ All installed!


In [37]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import os
import warnings
warnings.filterwarnings('ignore')

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from bs4 import BeautifulSoup

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.metrics.pairwise import cosine_similarity
import joblib

print("✅ All libraries imported!")

✅ All libraries imported!


In [38]:
# Load dataset
df = pd.read_csv('../data/raw/Resume.csv', encoding='utf-8')

print("=" * 50)
print("✅ DATASET LOADED!")
print("=" * 50)
print(f"Total Resumes  : {len(df)}")
print(f"Columns        : {df.columns.tolist()}")
print(f"Categories     : {df['Category'].nunique()}")
print()
print(df['Category'].value_counts())

✅ DATASET LOADED!
Total Resumes  : 3446
Columns        : ['ID', 'Resume_str', 'Resume_html', 'Category', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17', 'Unnamed: 18', 'Unnamed: 19', 'Unnamed: 20', 'Unnamed: 21', 'Unnamed: 22', 'Unnamed: 23', 'Unnamed: 24', 'Unnamed: 25', 'Unnamed: 26', 'Unnamed: 27', 'Unnamed: 28', 'Unnamed: 29', 'Unnamed: 30', 'Unnamed: 31', 'Unnamed: 32', 'Unnamed: 33', 'Unnamed: 34', 'Unnamed: 35', 'Unnamed: 36', 'Unnamed: 37', 'Unnamed: 38', 'Unnamed: 39', 'Unnamed: 40', 'Unnamed: 41', 'Unnamed: 42', 'Unnamed: 43', 'Unnamed: 44', 'Unnamed: 45', 'Unnamed: 46', 'Unnamed: 47', 'Unnamed: 48', 'Unnamed: 49', 'Unnamed: 50', 'Unnamed: 51', 'Unnamed: 52', 'Unnamed: 53', 'Unnamed: 54', 'Unnamed: 55', 'Unnamed: 56', 'Unnamed: 57', 'Unnamed: 58', 'Unnamed: 59', 'Unnamed: 60', 'Unnamed: 61', 'Unnamed: 62', 'Unnamed: 63', '

In [39]:
print("DATASET INFORMATION")
print("=" * 50)
print()

# Shape
print(f"Shape: {df.shape}")
print()

# Data types
print("Data Types:")
print(df.dtypes)
print()

# Missing values
print("Missing Values:")
print(df.isnull().sum())
print()

# Duplicate rows
print(f"Duplicate rows: {df.duplicated().sum()}")
print()

# Category counts
print("CATEGORY DISTRIBUTION:")
print("=" * 40)
cat_counts = df['Category'].value_counts()
print(cat_counts.to_string())
print()
print(f"Total unique categories: {df['Category'].nunique()}")

DATASET INFORMATION

Shape: (3446, 169)

Data Types:
ID              object
Resume_str      object
Resume_html     object
Category        object
Unnamed: 4      object
                 ...  
Unnamed: 164    object
Unnamed: 165    object
Unnamed: 166    object
Unnamed: 167    object
Unnamed: 168    object
Length: 169, dtype: object

Missing Values:
ID               115
Resume_str       732
Resume_html      825
Category         911
Unnamed: 4      3385
                ... 
Unnamed: 164    3444
Unnamed: 165    3445
Unnamed: 166    3445
Unnamed: 167    3445
Unnamed: 168    3445
Length: 169, dtype: int64

Duplicate rows: 736

CATEGORY DISTRIBUTION:
Category
INFORMATION-TECHNOLOGY                                                                                                                                                                                                                                                                                                                             

In [40]:
import pandas as pd
import numpy as np
import re
import os
import warnings
warnings.filterwarnings('ignore')

# Load raw CSV
df_raw = pd.read_csv('../data/raw/Resume.csv',
                     encoding='utf-8')

print("BEFORE CLEANING")
print("=" * 50)
print(f"Shape          : {df_raw.shape}")
print(f"Total rows     : {len(df_raw)}")
print(f"Total columns  : {len(df_raw.columns)}")
print()
print("First 4 columns:")
print(df_raw[['ID','Resume_str',
              'Resume_html','Category']].head(3))

BEFORE CLEANING
Shape          : (3446, 169)
Total rows     : 3446
Total columns  : 169

First 4 columns:
         ID                                         Resume_str  \
0  16852973           HR ADMINISTRATOR/MARKETING ASSOCIATE\...   
1  22323967           HR SPECIALIST, US HR OPERATIONS      ...   
2  33176873           HR DIRECTOR       Summary      Over 2...   

                                         Resume_html Category  
0  <div class="fontsize fontface vmargins hmargin...       HR  
1  <div class="fontsize fontface vmargins hmargin...       HR  
2  <div class="fontsize fontface vmargins hmargin...       HR  


In [41]:
# We only need these 4 columns
# Everything else (Unnamed columns) is garbage
df = df_raw[['ID', 'Resume_str',
             'Resume_html', 'Category']].copy()

print("KEPT ONLY 4 USEFUL COLUMNS")
print("=" * 50)
print(f"Shape now: {df.shape}")
print()
print("Nulls per column:")
print(df.isnull().sum())

KEPT ONLY 4 USEFUL COLUMNS
Shape now: (3446, 4)

Nulls per column:
ID             115
Resume_str     732
Resume_html    825
Category       911
dtype: int64


In [42]:
print("BEFORE:", len(df), "rows")

# Step 1: Drop duplicate rows
df = df.drop_duplicates()
print(f"After removing duplicates : {len(df)}")

# Step 2: Drop rows where Category is null
df = df.dropna(subset=['Category'])
print(f"After removing null Category: {len(df)}")

# Step 3: Drop rows where Resume_str is null
df = df.dropna(subset=['Resume_str'])
print(f"After removing null Resume  : {len(df)}")

# Step 4: Reset index
df.reset_index(drop=True, inplace=True)
print()
print("AFTER:", len(df), "rows")

BEFORE: 3446 rows
After removing duplicates : 2710
After removing null Category: 2521
After removing null Resume  : 2521

AFTER: 2521 rows


In [43]:
# These are the ONLY valid categories
VALID_CATEGORIES = [
    'INFORMATION-TECHNOLOGY', 'BUSINESS-DEVELOPMENT',
    'ADVOCATE',   'CHEF',          'FINANCE',
    'ENGINEERING','ACCOUNTANT',    'FITNESS',
    'AVIATION',   'SALES',         'HEALTHCARE',
    'CONSULTANT',  'BANKING',      'CONSTRUCTION',
    'PUBLIC-RELATIONS', 'HR',      'DESIGNER',
    'ARTS',       'TEACHER',       'APPAREL',
    'DIGITAL-MEDIA','AGRICULTURE', 'AUTOMOBILE', 'BPO'
]

print("BEFORE category fix:")
print(f"Unique categories: {df['Category'].nunique()}")
print()

# Clean category values
df['Category'] = df['Category'].astype(str).str.strip()
df['Category'] = df['Category'].str.upper()

# Keep ONLY rows with valid categories
df = df[df['Category'].isin(VALID_CATEGORIES)]
df.reset_index(drop=True, inplace=True)

print("AFTER category fix:")
print(f"Unique categories: {df['Category'].nunique()}")
print(f"Total rows left  : {len(df)}")
print()
print(df['Category'].value_counts().to_string())

BEFORE category fix:
Unique categories: 75

AFTER category fix:
Unique categories: 24
Total rows left  : 2458

Category
INFORMATION-TECHNOLOGY    119
BUSINESS-DEVELOPMENT      119
AVIATION                  117
CHEF                      117
FITNESS                   117
ACCOUNTANT                116
ENGINEERING               116
FINANCE                   116
BANKING                   115
ADVOCATE                  115
SALES                     115
CONSULTANT                115
HEALTHCARE                113
PUBLIC-RELATIONS          110
HR                        109
CONSTRUCTION              108
DESIGNER                  106
ARTS                      103
TEACHER                   102
APPAREL                    96
DIGITAL-MEDIA              95
AGRICULTURE                63
AUTOMOBILE                 35
BPO                        21


In [45]:
from bs4 import BeautifulSoup

def remove_html(text):
    """Remove all HTML tags from text."""
    text = str(text)
    # Method 1: BeautifulSoup
    try:
        text = BeautifulSoup(text,
                             'html.parser').get_text(
                             separator=' ')
    except:
        pass
    # Method 2: Regex backup
    text = re.sub(r'<[^>]+>', ' ', text)
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def clean_text(text):
    """Full text cleaning pipeline."""
    # Step 1: Remove HTML
    text = remove_html(text)
    # Step 2: Remove special characters
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    # Step 3: Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    # Step 4: Lowercase
    text = text.lower()
    # Step 5: Remove very short words
    words = [w for w in text.split() if len(w) > 2]
    return ' '.join(words)

# Test on one sample
sample = df['Resume_str'].iloc[1]
print("ORIGINAL (200 chars):")
print(sample[:200])
print()
print("CLEANED (200 chars):")
print(clean_text(sample)[:200])
print()
print("✅ Cleaning function works!")

ORIGINAL (200 chars):
         HR SPECIALIST, US HR OPERATIONS       Summary     Versatile  media professional with background in Communications, Marketing, Human Resources and Technology.         Experience     09/2015   

CLEANED (200 chars):
specialist operations summary versatile media professional with background communications marketing human resources and technology experience current specialist operations company name city state mana

✅ Cleaning function works!


In [46]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords', quiet=True)
nltk.download('wordnet',   quiet=True)
nltk.download('punkt',     quiet=True)

STOP_WORDS = set(stopwords.words('english'))
LEMMATIZER = WordNetLemmatizer()

# Words to always keep
KEEP_WORDS = {
    'python', 'java', 'sql', 'excel', 'aws',
    'gcp', 'azure', 'machine', 'learning', 'deep',
    'data', 'science', 'management', 'experience',
    'years', 'project', 'team', 'lead', 'analyst',
    'sales', 'finance', 'marketing', 'design',
    'engineering', 'medical', 'legal', 'teaching',
}

def full_clean(text):
    """Complete NLP cleaning pipeline."""
    # Basic clean first
    text   = clean_text(text)
    tokens = text.split()
    # Remove stopwords but keep skill words
    tokens = [
        LEMMATIZER.lemmatize(w)
        for w in tokens
        if (w not in STOP_WORDS or w in KEEP_WORDS)
        and len(w) > 2
    ]
    return ' '.join(tokens)

print("Cleaning all resumes...")
print("Please wait 3-5 minutes...")
print()

import time
start = time.time()

df['Clean_Resume'] = df['Resume_str'].apply(full_clean)

elapsed = time.time() - start
print(f"✅ Done in {elapsed:.1f} seconds!")
print(f"Total resumes cleaned: {len(df)}")

Cleaning all resumes...
Please wait 3-5 minutes...

✅ Done in 15.3 seconds!
Total resumes cleaned: 2458


In [47]:
print("Extracting useful info from resumes...")

# ── Email ─────────────────────────────────────────────────
def get_email(text):
    text  = str(text)
    match = re.search(
        r'[a-zA-Z0-9._%+-]+@'
        r'[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}',
        text)
    return match.group(0).lower() if match else ''

# ── Phone ─────────────────────────────────────────────────
def get_phone(text):
    text  = str(text)
    match = re.search(
        r'(\+?[\d][\d\s\-\(\)]{8,}[\d])', text)
    if match:
        ph = re.sub(r'[\s\-\(\)]', '',
                    match.group(0))
        return ph if len(ph) >= 10 else ''
    return ''

# ── Years Experience ──────────────────────────────────────
def get_years(text):
    text    = str(text)
    matches = re.findall(
        r'(\d+)\s*\+?\s*(?:years?|yrs?)'
        r'(?:\s+of)?\s*(?:experience|exp)?',
        text.lower())
    if matches:
        nums = [int(m) for m in matches
                if 0 < int(m) < 50]
        return max(nums) if nums else 0
    return 0

# ── Word Count ────────────────────────────────────────────
def get_word_count(text):
    return len(str(text).split())

# Apply all
df['Email']      = df['Resume_str'].apply(get_email)
df['Phone']      = df['Resume_str'].apply(get_phone)
df['Years_Exp']  = df['Resume_str'].apply(get_years)
df['Word_Count'] = df['Clean_Resume'].apply(
                       get_word_count)

print("=" * 50)
print("✅ EXTRACTION COMPLETE!")
print("=" * 50)
print(f"Emails found  : {(df['Email']!='').sum()}")
print(f"Phones found  : {(df['Phone']!='').sum()}")
print(f"Avg years exp : {df['Years_Exp'].mean():.1f}")
print(f"Avg word count: {df['Word_Count'].mean():.0f}")

Extracting useful info from resumes...
✅ EXTRACTION COMPLETE!
Emails found  : 19
Phones found  : 80
Avg years exp : 4.7
Avg word count: 572


In [48]:
print("BEFORE quality filter:", len(df))

# Remove resumes with too few words
df = df[df['Word_Count'] > 30]
print(f"After word count filter : {len(df)}")

# Remove resumes where clean text is too short
df = df[df['Clean_Resume'].str.len() > 50]
print(f"After text length filter: {len(df)}")

# Fill remaining nulls
df['Email']     = df['Email'].fillna('')
df['Phone']     = df['Phone'].fillna('')
df['Years_Exp'] = df['Years_Exp'].fillna(0).astype(int)
df['ID']        = df['ID'].fillna('unknown')

df.reset_index(drop=True, inplace=True)

print()
print("✅ QUALITY FILTER DONE!")
print(f"Final rows: {len(df)}")

BEFORE quality filter: 2458
After word count filter : 2457
After text length filter: 2457

✅ QUALITY FILTER DONE!
Final rows: 2457


In [49]:
print("=" * 55)
print("FINAL DATA QUALITY REPORT")
print("=" * 55)
print()
print(f"Total resumes    : {len(df)}")
print(f"Total categories : {df['Category'].nunique()}")
print()

print("NULL CHECK:")
print("-" * 40)
for col in ['ID','Category','Clean_Resume',
            'Email','Phone','Years_Exp',
            'Word_Count']:
    if col in df.columns:
        nulls  = df[col].isnull().sum()
        empty  = (df[col].astype(str)=='').sum()
        status = "✅" if nulls+empty == 0 else "⚠️ "
        print(f"  {status} {col:<20} "
              f"nulls:{nulls}  empty:{empty}")

print()
print("CATEGORY COUNTS:")
print("-" * 40)
print(df['Category'].value_counts().to_string())

FINAL DATA QUALITY REPORT

Total resumes    : 2457
Total categories : 24

NULL CHECK:
----------------------------------------
  ✅ ID                   nulls:0  empty:0
  ✅ Category             nulls:0  empty:0
  ✅ Clean_Resume         nulls:0  empty:0
  ⚠️  Email                nulls:0  empty:2438
  ⚠️  Phone                nulls:0  empty:2377
  ✅ Years_Exp            nulls:0  empty:0
  ✅ Word_Count           nulls:0  empty:0

CATEGORY COUNTS:
----------------------------------------
Category
INFORMATION-TECHNOLOGY    119
BUSINESS-DEVELOPMENT      118
AVIATION                  117
CHEF                      117
FITNESS                   117
ACCOUNTANT                116
ENGINEERING               116
FINANCE                   116
BANKING                   115
ADVOCATE                  115
SALES                     115
CONSULTANT                115
HEALTHCARE                113
PUBLIC-RELATIONS          110
HR                        109
CONSTRUCTION              108
DESIGNER             

In [50]:
df

,ID,Resume_str,Resume_html,Category,Clean_Resume,Email,Phone,Years_Exp,Word_Count
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR,administrator marketing associate administrato...,,,15,495
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR,specialist operation summary versatile medium ...,,,0,523
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR,director summary year experience recruiting pl...,,,20,701
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR,specialist summary dedicated driven dynamic ye...,,,20,259
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR,manager skill highlight skill department start...,,,0,845
...,...,...,...,...,...,...,...,...,...
2452,99416532,RANK: SGT/E-5 NON- COMMISSIONED OFFIC...,"<div class=""fontsize fontface vmargins hmargin...",AVIATION,rank sgt non commissioned officer charge briga...,,,0,471
2453,24589765,"GOVERNMENT RELATIONS, COMMUNICATIONS ...","<div class=""fontsize fontface vmargins hmargin...",AVIATION,government relation communication organization...,,,0,647
2454,31605080,GEEK SQUAD AGENT Professional...,"<div class=""fontsize fontface vmargins hmargin...",AVIATION,geek squad agent professional profile support ...,,,0,181
2455,21190805,PROGRAM DIRECTOR / OFFICE MANAGER ...,"<div class=""fontsize fontface vmargins hmargin...",AVIATION,program director office manager summary highly...,,,5,504


In [51]:
# Keep only these final columns
final_df = df[[
    'ID',
    'Category',
    'Clean_Resume',
    'Resume_str',
    'Email',
    'Phone',
    'Years_Exp',
    'Word_Count'
]].copy()

# Save
os.makedirs('../data/processed', exist_ok=True)
final_df.to_csv(
    '../data/processed/resumes_final.csv',
    index=False,
    encoding='utf-8')

print("=" * 55)
print("✅ CLEAN DATA SAVED!")
print("=" * 55)
print()
print("File : data/processed/resumes_final.csv")
print(f"Rows : {len(final_df)}")
print(f"Cols : {final_df.columns.tolist()}")
print()
print("Preview:")
print(final_df[['Category','Years_Exp',
                'Email','Word_Count']
               ].head(10).to_string())
print()
print("=" * 55)
print("NEXT STEP: Use resumes_final.csv")
print("           to train your model!")
print("=" * 55)

✅ CLEAN DATA SAVED!

File : data/processed/resumes_final.csv
Rows : 2457
Cols : ['ID', 'Category', 'Clean_Resume', 'Resume_str', 'Email', 'Phone', 'Years_Exp', 'Word_Count']

Preview:
  Category  Years_Exp Email  Word_Count
0       HR         15               495
1       HR          0               523
2       HR         20               701
3       HR         20               259
4       HR          0               845
5       HR          0               488
6       HR          0               482
7       HR          0               827
8       HR         15               598
9       HR          0               532

NEXT STEP: Use resumes_final.csv
           to train your model!
